In [19]:
import json
import pandas as pd
import numpy as np

# 1. Carregando os dados
with open('../dados/dados_nivel_1.json', 'r', encoding='utf-8') as f:
    dados_brutos = json.load(f)

taxa_cambio = dados_brutos['taxa_cambio_usd_brl']
df = pd.DataFrame(dados_brutos['operacoes'])

print(f"Taxa de Câmbio Fixa: {taxa_cambio}")
print("\n--- Informações do DataFrame ---")
df.info()

print("\n--- Amostra de Valores Únicos nas Colunas Categóricas ---")
print("Moedas:", df['moeda'].unique())
print("Canais:", df['canal'].unique())

print("\n--- Verificando Valores Nulos ou Negativos ---")
display(df[df.isnull().any(axis=1)])
display(df[df['valor'] <= 0])

Taxa de Câmbio Fixa: 5.4

--- Informações do DataFrame ---
<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   id           20 non-null     str  
 1   cliente_id   20 non-null     str  
 2   data         19 non-null     str  
 3   valor        20 non-null     int64
 4   moeda        20 non-null     str  
 5   canal        20 non-null     str  
 6   tipo         20 non-null     str  
 7   contraparte  20 non-null     str  
 8   observacao   20 non-null     str  
dtypes: int64(1), str(8)
memory usage: 2.9 KB

--- Amostra de Valores Únicos nas Colunas Categóricas ---
Moedas: <ArrowStringArray>
['BRL', 'USD']
Length: 2, dtype: str
Canais: <ArrowStringArray>
['pix', 'ted', 'boleto', 'cartao', 'especie']
Length: 5, dtype: str

--- Verificando Valores Nulos ou Negativos ---


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
17,OP-0017,CLI-A-5,NaN,4300,BRL,especie,deposito,Gama Distribuidora,data nao capturada pelo sistema


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao


### 2. Limpeza e Tratamento dos Dados

**O que eu encontrei:**  
Dando uma olhada nos dados, vi que tem um valor nulo (`NaN`) na coluna `data`. Isso aconteceu na operação `OP-0017` do cliente `CLI-A-5`. A própria coluna de observação dessa linha diz: "data nao capturada pelo sistema". 

**Como eu resolvi e por quê:**  
Eu decidi **excluir** essa linha usando o `dropna()`. 
Fiz isso porque a Regra 1 (de Fracionamento) pede para somar as operações que acontecem na *mesma data*. Se eu deixar uma operação sem data no DataFrame, o Pandas não vai conseguir agrupar direito e a regra vai falhar ou dar um resultado errado. Como é só uma operação com erro, achei mais seguro tirar ela da análise para não estragar a lógica principal do desafio.

In [20]:
# --- PASSO 2: Limpeza dos Dados ---
# Removendo linhas onde a data é nula, conforme justificado no Markdown
df = df.dropna(subset=['data']).copy()

# Garantindo que a coluna de data seja do tipo datetime para ordenação/agrupamento correto
df['data'] = pd.to_datetime(df['data'])

# --- PASSO 3: Normalização para BRL ---
df['valor_brl'] = np.where(df['moeda'] == 'USD', df['valor'] * taxa_cambio, df['valor'])

# --- PASSO 4: Agregações ---
volume_por_cliente = df.groupby('cliente_id')['valor_brl'].sum().reset_index()
operacoes_por_canal = df['canal'].value_counts().reset_index()

print("--- Volume Total por Cliente ---")
display(volume_por_cliente)
print("\n--- Quantidade de Operações por Canal ---")
display(operacoes_por_canal)

# --- PASSO 5: Implementação das Regras ---

# REGRA 1: Fracionamento 
# (Mesma data, >= 3 ops, soma > 50k, nenhuma >= 20k)
agregado_diario = df.groupby(['cliente_id', 'data'])['valor_brl'].agg(
    soma_diaria='sum', 
    qtd_operacoes='count', 
    maior_operacao='max'
).reset_index()

agregado_diario['flag_fracionamento'] = (
    (agregado_diario['soma_diaria'] > 50000) & 
    (agregado_diario['qtd_operacoes'] >= 3) & 
    (agregado_diario['maior_operacao'] < 20000)
)

# Cruzando a flag de volta para o dataframe original
df = df.merge(agregado_diario[['cliente_id', 'data', 'flag_fracionamento']], on=['cliente_id', 'data'], how='left')

# REGRA 2: Valor Atípico 
# (> 5x mediana do cliente, aplicar apenas a clientes com >= 4 ops)
stats_cliente = df.groupby('cliente_id')['valor_brl'].agg(
    mediana_cliente='median', 
    qtd_total_ops='count'
).reset_index()

df = df.merge(stats_cliente, on='cliente_id', how='left')

df['flag_atipico'] = (
    (df['valor_brl'] > (df['mediana_cliente'] * 5)) & 
    (df['qtd_total_ops'] >= 4)
)

# Limpando colunas auxiliares usadas na regra 2
df = df.drop(columns=['mediana_cliente', 'qtd_total_ops'])

# --- PASSO 6: Validação das Regras ---
print("\n--- Validação da Regra 1 (Fracionamento) ---")
# Filtramos apenas os clientes que tiveram movimentações no dia 2026-03-09 para ver o comportamento
validacao_r1 = df[df['data'] == '2026-03-09'][['cliente_id', 'valor_brl', 'flag_fracionamento']].sort_values('cliente_id')
display(validacao_r1)

--- Volume Total por Cliente ---


,cliente_id,valor_brl
0,CLI-A-1,57500.0
1,CLI-A-2,52900.0
2,CLI-A-3,65700.0
3,CLI-A-4,79500.0
4,CLI-A-5,12600.0
5,CLI-A-6,10200.0



--- Quantidade de Operações por Canal ---


,canal,count
0,pix,9
1,ted,5
2,boleto,3
3,cartao,2



--- Validação da Regra 1 (Fracionamento) ---


,cliente_id,valor_brl,flag_fracionamento
0,CLI-A-1,18100.0,True
1,CLI-A-1,17300.0,True
2,CLI-A-1,18800.0,True


In [21]:
import os
import time
import json
import google.generativeai as genai
from dotenv import load_dotenv
from IPython.display import display, Markdown # <-- Nova biblioteca para deixar bonito!

# 1. Carregar a chave de API do arquivo .env
load_dotenv()
CHAVE_API = os.getenv("GEMINI_API_KEY")

if not CHAVE_API:
    print("ERRO: Chave API não encontrada! Verifique seu arquivo .env")
else:
    genai.configure(api_key=CHAVE_API)
    
    # 2. Configurar o modelo 
    modelo_ia = genai.GenerativeModel('gemini-3.6-flash')
    
    try:
        # 3. Filtrar o cliente suspeito que nossa regra pegou (CLI-A-1)
        cliente_suspeito = "CLI-A-1"
        dados_cli_a1 = df[df['cliente_id'] == cliente_suspeito].copy()
        dados_cli_a1['data'] = dados_cli_a1['data'].dt.strftime('%Y-%m-%d')
        extrato_cliente = dados_cli_a1.to_dict(orient='records')
        
        # 4. Criando o Prompt
        prompt = f"""
        Você é um analista de prevenção à lavagem de dinheiro (PLD).
        Analise o seguinte extrato bancário do cliente {cliente_suspeito}:
        {json.dumps(extrato_cliente, indent=2)}
        
        Este cliente caiu na regra de Fracionamento (operações picadas para burlar o limite de R$ 50 mil).
        Me devolva a análise ESTRITAMENTE no formato JSON abaixo, sem texto antes ou depois:
        {{
          "nivel_risco": "baixo, medio ou alto",
          "tipologia_suspeita": "texto curto",
          "red_flags": ["lista de alertas"],
          "justificativa": "sua explicacao"
        }}
        """
        
        print("Iniciando a análise com o modelo gemini-3.6-flash...\n")
        inicio = time.time()
        
        # 5. Chamando a IA
        resposta = modelo_ia.generate_content(prompt)
        
        tempo_gasto = time.time() - inicio
        
        # 6. Tratando o retorno (Limpando formatação Markdown)
        texto_resposta = resposta.text.strip()
        if texto_resposta.startswith("```json"):
            texto_resposta = texto_resposta[7:-3]
        elif texto_resposta.startswith("```"):
            texto_resposta = texto_resposta[3:-3]
        
        # 7. Validando o JSON e criando um visual bonito
        try:
            analise_json = json.loads(texto_resposta)
            
            # Escolhendo a cor da bolinha baseada no risco
            nivel = analise_json.get('nivel_risco', '').lower()
            icone_risco = "🔴" if nivel == "alto" else "🟡" if nivel == "medio" else "🟢"
            
            # Montando o texto em Markdown formatado
            relatorio = f"""
### 🚨 Relatório de Análise PLD - Cliente: `{cliente_suspeito}`

**Nível de Risco:** {icone_risco} **{nivel.upper()}**  
**Tipologia Suspeita:** {analise_json.get('tipologia_suspeita')}

#### 🚩 Red Flags Identificadas:
"""
            for flag in analise_json.get('red_flags', []):
                relatorio += f"- {flag}\n"
                
            relatorio += f"""
#### 📋 Justificativa da IA:
*{analise_json.get('justificativa')}*

---
*⏱️ Tempo de resposta: {tempo_gasto:.2f} segundos* | *🪙 Tokens (In: {resposta.usage_metadata.prompt_token_count} | Out: {resposta.usage_metadata.candidates_token_count} | Total: {resposta.usage_metadata.total_token_count})*
"""
            # Exibindo na tela de forma visual
            display(Markdown(relatorio))
            
        except json.JSONDecodeError:
            print("Erro: A IA não devolveu um JSON válido.")
            print("Resposta bruta:", texto_resposta)
            
    except NameError:
        print("\n❌ ERRO: A tabela 'df' não foi encontrada na memória.")
        print("👉 SOLUÇÃO: Rode a célula da 'Parte A' e depois rode esta novamente!")

Iniciando a análise com o modelo gemini-3.6-flash...




### 🚨 Relatório de Análise PLD - Cliente: `CLI-A-1`

**Nível de Risco:** 🔴 **ALTO**  
**Tipologia Suspeita:** Fracionamento de operações (Smurfing)

#### 🚩 Red Flags Identificadas:
- Múltiplas transferências no mesmo dia (09/03/2026) com valores individualmente abaixo do patamar de alerta, mas que somadas somam R$ 54.200,00 (superando o limite de R$ 50.000,00)
- Transferências fracionadas enviadas no mesmo dia para a mesma contraparte (Alfa Comercio LTDA)
- Uso de diferentes canais de pagamento (PIX e TED) em um único dia para pulverizar as movimentações

#### 📋 Justificativa da IA:
*No dia 09/03/2026, o cliente realizou três operações enviadas (duas via PIX para Alfa Comercio LTDA e uma via TED para Beta Servicos ME) em valores aproximados entre R$ 17.300,00 e R$ 18.800,00. A soma total repassada no dia foi de R$ 54.200,00. O padrão de fracionar envios no mesmo dia para a mesma contraparte e utilizar múltiplos canais configura forte indício de tentativa intencional de burlar os parâmetros de monitoramento e os limites de comunicação compulsória ao Coaf.*

---
*⏱️ Tempo de resposta: 13.71 segundos* | *🪙 Tokens (In: 747 | Out: 324 | Total: 2487)*


In [ ]:
import os
import time
import json
import google.generativeai as genai
from dotenv import load_dotenv
from IPython.display import display, Markdown # <-- Nova biblioteca para deixar bonito!

# 1. Carregar a chave de API do arquivo .env
load_dotenv()
CHAVE_API = os.getenv("GEMINI_API_KEY")

if not CHAVE_API:
    print("ERRO: Chave API não encontrada! Verifique seu arquivo .env")
else:
    genai.configure(api_key=CHAVE_API)
    
    # 2. Configurar o modelo 
    modelo_ia = genai.GenerativeModel('gemini-3.6-flash')
    
    try:
        # 3. Filtrar o cliente suspeito que nossa regra pegou (CLI-A-1)
        cliente_suspeito = "CLI-A-1"
        dados_cli_a1 = df[df['cliente_id'] == cliente_suspeito].copy()
        dados_cli_a1['data'] = dados_cli_a1['data'].dt.strftime('%Y-%m-%d')
        extrato_cliente = dados_cli_a1.to_dict(orient='records')
        
        # 4. Criando o Prompt
        prompt = f"""
        Você é um analista de prevenção à lavagem de dinheiro (PLD).
        Analise o seguinte extrato bancário do cliente {cliente_suspeito}:
        {json.dumps(extrato_cliente, indent=2)}
        
        Este cliente caiu na regra de Fracionamento (operações picadas para burlar o limite de R$ 50 mil).
        Me devolva a análise ESTRITAMENTE no formato JSON abaixo, sem texto antes ou depois:
        {{
          "nivel_risco": "baixo, medio ou alto",
          "tipologia_suspeita": "texto curto",
          "red_flags": ["lista de alertas"],
          "justificativa": "sua explicacao"
        }}
        """
        
        print("Iniciando a análise com o modelo gemini-3.6-flash...\n")
        inicio = time.time()
        
        # 5. Chamando a IA
        resposta = modelo_ia.generate_content(prompt)
        
        tempo_gasto = time.time() - inicio
        
        # 6. Tratando o retorno (Limpando formatação Markdown)
        texto_resposta = resposta.text.strip()
        if texto_resposta.startswith("```json"):
            texto_resposta = texto_resposta[7:-3]
        elif texto_resposta.startswith("```"):
            texto_resposta = texto_resposta[3:-3]
        
        # 7. Validando o JSON e criando um visual bonito
        try:
            analise_json = json.loads(texto_resposta)
            
            # Escolhendo a cor da bolinha baseada no risco
            nivel = analise_json.get('nivel_risco', '').lower()
            icone_risco = "🔴" if nivel == "alto" else "🟡" if nivel == "medio" else "🟢"
            
            # Montando o texto em Markdown formatado
            relatorio = f"""
### 🚨 Relatório de Análise PLD - Cliente: `{cliente_suspeito}`

**Nível de Risco:** {icone_risco} **{nivel.upper()}**  
**Tipologia Suspeita:** {analise_json.get('tipologia_suspeita')}

#### 🚩 Red Flags Identificadas:
"""
            for flag in analise_json.get('red_flags', []):
                relatorio += f"- {flag}\n"
                
            relatorio += f"""
#### 📋 Justificativa da IA:
*{analise_json.get('justificativa')}*

---
*⏱️ Tempo de resposta: {tempo_gasto:.2f} segundos* | *🪙 Tokens (In: {resposta.usage_metadata.prompt_token_count} | Out: {resposta.usage_metadata.candidates_token_count} | Total: {resposta.usage_metadata.total_token_count})*
"""
            # Exibindo na tela de forma visual
            display(Markdown(relatorio))
            
        except json.JSONDecodeError:
            print("Erro: A IA não devolveu um JSON válido.")
            print("Resposta bruta:", texto_resposta)
            
    except NameError:
        print("\n❌ ERRO: A tabela 'df' não foi encontrada na memória.")
        print("👉 SOLUÇÃO: Rode a célula da 'Parte A' e depois rode esta novamente!")

Iniciando a análise com o modelo gemini-3.6-flash...




### 🚨 Relatório de Análise PLD - Cliente: `CLI-A-1`

**Nível de Risco:** 🔴 **ALTO**  
**Tipologia Suspeita:** Fracionamento de operações (Smurfing)

#### 🚩 Red Flags Identificadas:
- Múltiplas transferências no mesmo dia (09/03/2026) com valores individualmente abaixo do patamar de alerta, mas que somadas somam R$ 54.200,00 (superando o limite de R$ 50.000,00)
- Transferências fracionadas enviadas no mesmo dia para a mesma contraparte (Alfa Comercio LTDA)
- Uso de diferentes canais de pagamento (PIX e TED) em um único dia para pulverizar as movimentações

#### 📋 Justificativa da IA:
*No dia 09/03/2026, o cliente realizou três operações enviadas (duas via PIX para Alfa Comercio LTDA e uma via TED para Beta Servicos ME) em valores aproximados entre R$ 17.300,00 e R$ 18.800,00. A soma total repassada no dia foi de R$ 54.200,00. O padrão de fracionar envios no mesmo dia para a mesma contraparte e utilizar múltiplos canais configura forte indício de tentativa intencional de burlar os parâmetros de monitoramento e os limites de comunicação compulsória ao Coaf.*

---
*⏱️ Tempo de resposta: 13.71 segundos* | *🪙 Tokens (In: 747 | Out: 324 | Total: 2487)*


### 💡 Avaliação do Parecer Gerado

* **Precisão Fática:** A LLM não alucinou valores nem datas; todos os números citados no parecer técnico coincidem estritamente com as agregações prévias do Pandas.
* **Qualidade Regulatória:** Identificação correta da tipologia de *Smurfing/Structuring*, contextualizando o fracionamento intencional abaixo da régua de R$ 50 mil.
* **Prontidão para Produção:** O retorno estrito em JSON viabiliza o consumo do parecer por filas de mensageria, sistemas legados de investigação ou envio automatizado de Comunicações de Operações Suspeitas (COS) ao COAF.